# LAPD NIBRS cleaning walkthrough

This notebook follows the class workflow:

**Envision → Plan → Set rules → Make domain decisions → Implement → Validate → Refine**

The purpose is to understand every cleaning decision across both professor-provided core datasets and connect the evidence to the team's Data Fit assessment. The two sources are cleaned separately because their rows and offense structures differ. Run one cell at a time. Read the interpretation after each result before continuing.

> Kernel: select the Python environment from `06-decompose/.venv`, which already contains pandas, Plotly, and PyArrow.

## 1. Envision: start with the goal

**Goal:** Prepare the Current LAPD NIBRS Offenses file and the Legacy LA Crime Data 2020–2024 file for transparent analysis and dashboard prototyping while preserving what a row means in each source. Use the cleaning evidence to evaluate two candidate decisions:

- **Vision A:** prioritize geographic areas and time periods for a pilot expansion of non-enforcement community outreach resources.
- **Vision B:** prioritize geographic areas with recurring nighttime reported offenses for on-site lighting assessments.

Success does not mean that every missing or suspicious value disappears. Success means that transformations are documented, questionable values stay visible, each cleaned table preserves its own source meaning, and the two reporting systems are not combined until their comparability is demonstrated.

## 2. Plan before coding

Open [`plan.md`](plan.md) before running the cleaning section. It contains proposed rules for all 31 NIBRS columns, all 28 Legacy columns, and the cross-dataset comparability decision.

The rules used here are:

1. Preserve both downloaded CSVs; keep the NIBRS raw table unchanged and process Legacy from unchanged source chunks.
2. Do not drop rows automatically.
3. Preserve identifiers and codes as text.
4. Flag suspicious values before deciding whether a particular analysis should exclude them.
5. Report row counts, missing values, duplicates, conversion failures, and changed cells.
6. Distinguish NIBRS offense records (`UniqueNIBRNo`), NIBRS cases (`CaseNo`), and Legacy records (`DR_NO`).
7. Treat the data as reported offenses, not all offenses that occurred.
8. Do not append the two files or compare their raw row counts as one uninterrupted trend without a validated crosswalk and overlap rule.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 90)

NIBRS_FILENAME = "export.csv%3FaccessType=DOWNLOAD.csv"
LEGACY_FILENAME = "export.csv%3FaccessType=DOWNLOAD (1).csv"
data_directories = [Path("../dataset"), Path("dataset")]
DATA_PATH = next((folder / NIBRS_FILENAME).resolve() for folder in data_directories if (folder / NIBRS_FILENAME).exists())
LEGACY_DATA_PATH = next((folder / LEGACY_FILENAME).resolve() for folder in data_directories if (folder / LEGACY_FILENAME).exists())

print(f"NIBRS source:  {DATA_PATH}")
print(f"Legacy source: {LEGACY_DATA_PATH}")

In [ ]:
# Load every source column as text first. This preserves leading zeros in identifiers and codes.
raw = pd.read_csv(
    DATA_PATH,
    dtype="string[pyarrow]",
    keep_default_na=False,
    na_values=[""],
    low_memory=False,
)

print(f"NIBRS: {len(raw):,} rows × {raw.shape[1]} columns")
print("Legacy will be loaded after the NIBRS validation to control memory use.")
display(raw.head(3))

## 3. Inspect before changing anything

One row is one **offense record**. It is not automatically one incident, one victim, or one crime location. The first inspection checks missingness, uniqueness, examples, and the difference between offense rows and cases.

In [ ]:
profile = pd.DataFrame({
    "dtype_loaded": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_pct": raw.isna().mean().mul(100).round(2),
    "unique_nonmissing": raw.nunique(dropna=True),
})
profile["examples"] = [
    raw[column].dropna().astype(str).drop_duplicates().head(3).tolist()
    for column in raw.columns
]
display(profile)

In [ ]:
missing_plot = (
    profile.loc[profile["missing"].gt(0), ["missing", "missing_pct"]]
    .sort_values("missing_pct")
    .reset_index(names="column")
)
fig = px.bar(
    missing_plot,
    x="missing_pct",
    y="column",
    orientation="h",
    title="Missing values in the downloaded NIBRS file",
    labels={"missing_pct": "Missing rows (%)", "column": ""},
)
fig.show()

In [ ]:
grain_check = pd.DataFrame({
    "measure": ["Offense rows", "Distinct UniqueNIBRNo", "Distinct CaseNo", "Rows in multi-offense cases"],
    "value": [
        len(raw),
        raw["UniqueNIBRNo"].nunique(dropna=True),
        raw["CaseNo"].nunique(dropna=True),
        raw["CaseNo"].duplicated(keep=False).sum(),
    ],
})
display(grain_check.style.format({"value": "{:,}"}))

fig = px.bar(
    grain_check.iloc[:3],
    x="measure",
    y="value",
    title="Why the counting unit matters",
    labels={"value": "Count", "measure": ""},
)
fig.show()

### Interpretation: why this appears in Data Fit

`UniqueNIBRNo` identifies an offense row. `CaseNo` identifies a case that can contain more than one offense. A dashboard that counts rows answers “How many offense records?” A dashboard that counts distinct `CaseNo` values answers “How many recorded cases?” Those are different measures, so Data Fit explicitly says the team must label the counting unit.

In [ ]:
date_occ_preview = pd.to_datetime(
    raw["Date OCC"], format="%Y %b %d %I:%M:%S %p", errors="coerce"
)
date_rptd_preview = pd.to_datetime(
    raw["Date Rptd"], format="%Y %b %d %I:%M:%S %p", errors="coerce"
)
time_numeric_preview = pd.to_numeric(raw["Time OCC"], errors="coerce")
time_valid_preview = (
    raw["Time OCC"].str.len().eq(4)
    & raw["Time OCC"].str.isdigit()
    & time_numeric_preview.floordiv(100).between(0, 23)
    & time_numeric_preview.mod(100).between(0, 59)
)
lat_preview = pd.to_numeric(raw["LAT"], errors="coerce")
lon_preview = pd.to_numeric(raw["LON"], errors="coerce")

quality_evidence = pd.DataFrame({
    "check": [
        "Date OCC parse failures",
        "Date Rptd missing or parse failures",
        "Occurrence dates before 2000",
        "Invalid Time OCC values",
        "Time OCC equal to 0000",
        "Missing latitude/longitude",
        "Zero latitude or longitude",
    ],
    "rows": [
        date_occ_preview.isna().sum(),
        date_rptd_preview.isna().sum(),
        date_occ_preview.lt("2000-01-01").sum(),
        (~time_valid_preview).sum(),
        raw["Time OCC"].eq("0000").sum(),
        (lat_preview.isna() | lon_preview.isna()).sum(),
        (lat_preview.eq(0) | lon_preview.eq(0)).sum(),
    ],
})
display(quality_evidence.style.format({"rows": "{:,}"}))

In [ ]:
year_counts = (
    date_occ_preview.loc[date_occ_preview.dt.year.ge(2000)]
    .dt.year.value_counts().sort_index().rename_axis("year").reset_index(name="offense_records")
)
px.line(
    year_counts, x="year", y="offense_records", markers=True,
    title="Occurrence-year distribution (years 2000 and later)"
).show()

premise_counts = (
    raw["Premise Desc"].fillna("Missing").value_counts().head(15)
    .sort_values().rename_axis("premise").reset_index(name="offense_records")
)
px.bar(
    premise_counts, x="offense_records", y="premise", orientation="h",
    title="Most common reported premise categories"
).show()

## 4. Human decisions before implementation

The student reviewed the evidence and confirmed the settings below.

**HUMAN DECISION 1 — nighttime (approved):** Use 8:00 p.m. through 5:59 a.m. Label it as a clock-time definition, not proof that a location was dark.

**HUMAN DECISION 2 — counting unit (approved):** Use distinct NIBRS `CaseNo` as the primary metric and offense records as a secondary metric. Show Legacy record counts separately.

**HUMAN DECISION 3 — date scope:** Cleaning retains all dates and flags dates before 2000. A later dashboard may use a documented date filter, but that filter is an analysis decision rather than a silent cleaning step.

**HUMAN DECISION 4 — geography:** LAPD area or reporting district is safer for comparison than treating privacy-rounded coordinates as exact incident points.

In [ ]:
# Edit these values only after reviewing the evidence and discussing the definitions.
NIGHT_START_HOUR = 20  # 8:00 p.m.
NIGHT_END_HOUR = 6     # before 6:00 a.m.
EARLY_DATE_REVIEW_CUTOFF = pd.Timestamp("2000-01-01")
SAVE_CLEAN_DATA = False  # keep False while learning and reviewing

print(f"Nighttime rule: hour >= {NIGHT_START_HOUR} or hour < {NIGHT_END_HOUR}")
print(f"Early-date review cutoff: {EARLY_DATE_REVIEW_CUTOFF.date()}")

## 5. Implement the approved rules

The code below creates a separate `clean` table. It trims text, converts documented date and numeric fields, and adds review flags. It does not delete any source row or fill unknown values with invented values.

In [ ]:
clean = raw.copy()
source_columns = raw.columns.tolist()

# 1. Trim outer whitespace in text fields and record exactly how many cells changed.
trim_change_counts = {}
trim_change_masks = {}
for column in source_columns:
    before = clean[column].copy()
    after = before.str.strip()
    changed = before.fillna("<MISSING>").ne(after.fillna("<MISSING>"))
    trim_change_counts[column] = int(changed.sum())
    trim_change_masks[column] = changed
    clean[column] = after

# 2. Convert fields whose source meanings are numeric or datetime.
date_columns = ["Date Rptd", "Date OCC"]
for column in date_columns:
    clean[column] = pd.to_datetime(
        clean[column], format="%Y %b %d %I:%M:%S %p", errors="coerce"
    )

numeric_columns = ["TotalOffenseCount", "TotalVictimCount", "LAT", "LON"]
for column in numeric_columns:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")

# 3. Validate time without replacing the source text.
time_numeric = pd.to_numeric(clean["Time OCC"], errors="coerce")
time_valid = (
    clean["Time OCC"].str.len().eq(4)
    & clean["Time OCC"].str.isdigit()
    & time_numeric.floordiv(100).between(0, 23)
    & time_numeric.mod(100).between(0, 59)
)
clean["Hour OCC"] = time_numeric.floordiv(100).where(time_valid).astype("Int64")
clean["time_invalid_flag"] = ~time_valid
clean["time_0000_flag"] = clean["Time OCC"].eq("0000")

# 4. Add review flags instead of deleting suspicious records.
clean["early_date_review_flag"] = clean["Date OCC"].lt(EARLY_DATE_REVIEW_CUTOFF).fillna(False)
clean["coordinate_issue_flag"] = (
    clean["LAT"].isna()
    | clean["LON"].isna()
    | clean["LAT"].eq(0)
    | clean["LON"].eq(0)
    | ~clean["LAT"].between(-90, 90)
    | ~clean["LON"].between(-180, 180)
)

# 5. Apply the editable nighttime definition.
if NIGHT_START_HOUR > NIGHT_END_HOUR:
    night_mask = clean["Hour OCC"].ge(NIGHT_START_HOUR) | clean["Hour OCC"].lt(NIGHT_END_HOUR)
else:
    night_mask = clean["Hour OCC"].between(NIGHT_START_HOUR, NIGHT_END_HOUR, inclusive="left")
clean["nighttime_flag"] = night_mask.fillna(False)

print(f"Source rows: {len(raw):,}")
print(f"Clean rows:  {len(clean):,}")
print(f"Cells changed only by trimming outer whitespace: {sum(trim_change_counts.values()):,}")

In [ ]:
trim_audit = (
    pd.Series(trim_change_counts, name="changed_cells")
    .loc[lambda values: values.gt(0)]
    .sort_values()
    .rename_axis("column")
    .reset_index()
)
display(trim_audit)
px.bar(
    trim_audit, x="changed_cells", y="column", orientation="h",
    title="Documented changes: cells with outer whitespace removed"
).show()

examples = []
for column in trim_audit["column"]:
    indices = trim_change_masks[column].loc[lambda values: values].index[:2]
    for index in indices:
        examples.append({
            "row_index": index,
            "column": column,
            "before": raw.at[index, column],
            "after": clean.at[index, column],
        })
display(pd.DataFrame(examples).head(10))

## 6. Validate: do not stop because the code ran

The next cells verify row preservation, identifier integrity, missingness, distributions, and actual before/after values. A failed check is evidence to investigate, not something to hide.

In [ ]:
validation = pd.DataFrame({
    "check": [
        "Row count preserved",
        "All source columns preserved",
        "UniqueNIBRNo has no missing values",
        "UniqueNIBRNo remains unique",
        "Source row order preserved",
        "CaseNo remains text",
        "Time OCC remains text",
    ],
    "passed": [
        len(clean) == len(raw),
        set(source_columns).issubset(clean.columns),
        clean["UniqueNIBRNo"].notna().all(),
        clean["UniqueNIBRNo"].is_unique,
        clean["UniqueNIBRNo"].reset_index(drop=True).equals(raw["UniqueNIBRNo"].str.strip().reset_index(drop=True)),
        str(clean["CaseNo"].dtype).startswith("string"),
        str(clean["Time OCC"].dtype).startswith("string"),
    ],
})
display(validation)
assert validation["passed"].all(), "Investigate failed validation checks before continuing."

In [ ]:
important_columns = [
    "UniqueNIBRNo", "CaseNo", "Date Rptd", "Date OCC", "Time OCC",
    "NIBR Code", "Crime Against", "Premise Desc", "LOCATION", "LAT", "LON",
]
missing_comparison = pd.DataFrame({
    "column": important_columns,
    "raw_missing": [raw[column].isna().sum() for column in important_columns],
    "clean_missing": [clean[column].isna().sum() for column in important_columns],
}).melt(id_vars="column", var_name="stage", value_name="missing_rows")

px.bar(
    missing_comparison, x="column", y="missing_rows", color="stage", barmode="group",
    title="Missing values before and after cleaning"
).show()

dtype_comparison = pd.DataFrame({
    "raw_dtype": raw[important_columns].dtypes.astype(str),
    "clean_dtype": clean[important_columns].dtypes.astype(str),
})
display(dtype_comparison)

In [ ]:
hour_summary = (
    clean.groupby("Hour OCC", dropna=False)
    .agg(offense_records=("UniqueNIBRNo", "count"), distinct_cases=("CaseNo", "nunique"))
    .reset_index()
)
hour_summary["period"] = hour_summary["Hour OCC"].apply(
    lambda hour: "Nighttime rule" if pd.notna(hour) and (hour >= NIGHT_START_HOUR or hour < NIGHT_END_HOUR) else "Other hours"
)
px.bar(
    hour_summary, x="Hour OCC", y="offense_records", color="period",
    title="Reported offense records by occurrence hour"
).show()

flag_summary = pd.DataFrame({
    "flag": ["Early occurrence date", "Coordinate issue", "Time equal to 0000", "Nighttime rule"],
    "rows": [
        clean["early_date_review_flag"].sum(),
        clean["coordinate_issue_flag"].sum(),
        clean["time_0000_flag"].sum(),
        clean["nighttime_flag"].sum(),
    ],
})
display(flag_summary.style.format({"rows": "{:,}"}))

valid_coordinates = clean.loc[~clean["coordinate_issue_flag"]]
coordinate_sample = valid_coordinates.sample(min(5_000, len(valid_coordinates)), random_state=576)
px.scatter(
    coordinate_sample, x="LON", y="LAT", color="AREA NAME",
    hover_data=["RPT Dist No", "Premise Desc"], opacity=0.45,
    title="Sample of usable privacy-rounded coordinates (not exact incident points)"
).show()

### Memory checkpoint before loading Legacy

The two CSVs contain about 1.48 million rows in total. To avoid keeping four large copies in memory, the next cell stores the NIBRS validation and comparison summaries, optionally saves the approved NIBRS table, and releases the large NIBRS DataFrames before loading Legacy. This is a memory-management step, not a deletion from the source file.

In [ ]:
import gc

nibrs_rows_raw = len(raw)
nibrs_rows_clean = len(clean)
nibrs_validation_result = validation.copy()
nibrs_years_for_comparison = (
    clean.loc[clean["Date OCC"].dt.year.between(2020, 2026), "Date OCC"]
    .dt.year.value_counts().sort_index().rename_axis("year").reset_index(name="rows")
)
nibrs_years_for_comparison["source"] = "NIBRS offense rows"
nibrs_2024_rows = int(clean["Date OCC"].dt.year.eq(2024).sum())
nibrs_case_id_set = set(clean["CaseNo"].dropna().astype(str))

if SAVE_CLEAN_DATA:
    nibrs_output_path = DATA_PATH.parent / "nibrs_clean.parquet"
    clean.to_parquet(nibrs_output_path, index=False)
    print(f"Saved {len(clean):,} NIBRS rows to {nibrs_output_path}")

del raw, clean
gc.collect()
print("NIBRS summaries retained; large NIBRS tables released before loading Legacy.")

## 7. Inspect and clean the second core dataset: Legacy 2020–2024

Legacy is not an unnecessary file. It supplies historical context that NIBRS does not. It must remain separate because its grain and offense structure differ: one Legacy row has a unique `DR_NO` and may store additional offenses in `Crm Cd 2–4`, while one NIBRS row is one offense identified by `UniqueNIBRNo`.

Because Legacy has about one million rows, the code below reads and cleans it in chunks. This is the same documented cleaning logic, but it avoids keeping two full copies in memory.

In [ ]:
from collections import Counter

LEGACY_CHUNKSIZE = 100_000
legacy_rows_raw = 0
legacy_rows_clean = 0
legacy_source_columns = None
legacy_missing_counts = None
legacy_trim_change_counts = None
legacy_seen_ids = set()
legacy_duplicate_id_values = 0
legacy_missing_id_rows = 0
legacy_invalid_occ_dates = 0
legacy_invalid_rpt_dates = 0
legacy_invalid_time_rows = 0
legacy_time_0000_rows = 0
legacy_coordinate_issue_rows = 0
legacy_rows_after_transition = 0
legacy_additional_code_counts = Counter()
legacy_year_counter = Counter()
legacy_hour_counter = Counter()
legacy_premise_counter = Counter()
legacy_clean_samples = []
identifier_matches = 0

legacy_chunks = pd.read_csv(
    LEGACY_DATA_PATH, dtype="string[pyarrow]", keep_default_na=False,
    na_values=[""], chunksize=LEGACY_CHUNKSIZE, low_memory=False,
)

for chunk_number, legacy_chunk_raw in enumerate(legacy_chunks, start=1):
    legacy_rows_raw += len(legacy_chunk_raw)
    if legacy_source_columns is None:
        legacy_source_columns = legacy_chunk_raw.columns.tolist()
        legacy_missing_counts = pd.Series(0, index=legacy_source_columns, dtype="int64")
        legacy_trim_change_counts = pd.Series(0, index=legacy_source_columns, dtype="int64")

    legacy_missing_counts = legacy_missing_counts.add(legacy_chunk_raw.isna().sum(), fill_value=0).astype("int64")
    legacy_chunk = legacy_chunk_raw.copy()
    for column in legacy_source_columns:
        before = legacy_chunk[column]
        after = before.str.strip()
        legacy_trim_change_counts[column] += int(before.fillna("<MISSING>").ne(after.fillna("<MISSING>")).sum())
        legacy_chunk[column] = after

    ids = legacy_chunk["DR_NO"].dropna().astype(str)
    legacy_missing_id_rows += int(legacy_chunk["DR_NO"].isna().sum())
    legacy_duplicate_id_values += int(ids.duplicated().sum()) + len(set(ids) & legacy_seen_ids)
    legacy_seen_ids.update(ids)
    identifier_matches += int(ids.isin(nibrs_case_id_set).sum())

    legacy_chunk["Date Rptd"] = pd.to_datetime(legacy_chunk["Date Rptd"], format="%Y %b %d %I:%M:%S %p", errors="coerce")
    legacy_chunk["DATE OCC"] = pd.to_datetime(legacy_chunk["DATE OCC"], format="%Y %b %d %I:%M:%S %p", errors="coerce")
    for column in ["Vict Age", "LAT", "LON"]:
        legacy_chunk[column] = pd.to_numeric(legacy_chunk[column], errors="coerce")

    legacy_time_numeric = pd.to_numeric(legacy_chunk["TIME OCC"], errors="coerce")
    legacy_time_valid = (
        legacy_chunk["TIME OCC"].str.len().eq(4)
        & legacy_chunk["TIME OCC"].str.isdigit()
        & legacy_time_numeric.floordiv(100).between(0, 23)
        & legacy_time_numeric.mod(100).between(0, 59)
    )
    legacy_chunk["Hour OCC"] = legacy_time_numeric.floordiv(100).where(legacy_time_valid).astype("Int64")
    legacy_chunk["time_invalid_flag"] = ~legacy_time_valid
    legacy_chunk["coordinate_issue_flag"] = (
        legacy_chunk["LAT"].isna() | legacy_chunk["LON"].isna()
        | legacy_chunk["LAT"].eq(0) | legacy_chunk["LON"].eq(0)
        | ~legacy_chunk["LAT"].between(-90, 90) | ~legacy_chunk["LON"].between(-180, 180)
    )
    legacy_night_mask = legacy_chunk["Hour OCC"].ge(NIGHT_START_HOUR) | legacy_chunk["Hour OCC"].lt(NIGHT_END_HOUR)
    legacy_chunk["nighttime_flag"] = legacy_night_mask.fillna(False)
    legacy_chunk["additional_crime_code_flag"] = legacy_chunk[["Crm Cd 2", "Crm Cd 3", "Crm Cd 4"]].notna().any(axis=1)

    legacy_invalid_occ_dates += int(legacy_chunk["DATE OCC"].isna().sum())
    legacy_invalid_rpt_dates += int(legacy_chunk["Date Rptd"].isna().sum())
    legacy_invalid_time_rows += int((~legacy_time_valid).sum())
    legacy_time_0000_rows += int(legacy_chunk["TIME OCC"].eq("0000").sum())
    legacy_coordinate_issue_rows += int(legacy_chunk["coordinate_issue_flag"].sum())
    legacy_rows_after_transition += int(legacy_chunk["DATE OCC"].ge("2024-03-07").sum())
    for column in ["Crm Cd 2", "Crm Cd 3", "Crm Cd 4"]:
        legacy_additional_code_counts[column] += int(legacy_chunk[column].notna().sum())
    legacy_year_counter.update(legacy_chunk["DATE OCC"].dt.year.dropna().astype(int).tolist())
    legacy_hour_counter.update(legacy_chunk["Hour OCC"].dropna().astype(int).tolist())
    legacy_premise_counter.update(legacy_chunk["Premis Desc"].dropna().tolist())
    if len(legacy_clean_samples) < 5:
        legacy_clean_samples.extend(legacy_chunk.head(5 - len(legacy_clean_samples)).to_dict(orient="records"))
    legacy_rows_clean += len(legacy_chunk)
    if chunk_number % 5 == 0:
        print(f"Processed {legacy_rows_clean:,} Legacy rows...")

legacy_clean_sample = pd.DataFrame(legacy_clean_samples)
legacy_profile = pd.DataFrame({
    "missing": legacy_missing_counts,
    "missing_pct": legacy_missing_counts.div(legacy_rows_raw).mul(100).round(2),
})
print(f"Legacy: {legacy_rows_raw:,} rows x {len(legacy_source_columns)} columns")

In [ ]:
display(legacy_profile)

legacy_missing_plot = (
    legacy_profile.loc[legacy_profile["missing"].gt(0), ["missing", "missing_pct"]]
    .sort_values("missing_pct").reset_index(names="column")
)
px.bar(
    legacy_missing_plot, x="missing_pct", y="column", orientation="h",
    title="Missing values in the Legacy 2020–2024 file",
    labels={"missing_pct": "Missing rows (%)", "column": ""},
).show()

legacy_quality = pd.DataFrame({
    "check": [
        "Rows", "Distinct DR_NO", "Duplicate DR_NO values", "Invalid occurrence dates",
        "Invalid TIME OCC", "Coordinate issue rows", "Rows with Crm Cd 2",
        "Rows with Crm Cd 3", "Rows with Crm Cd 4",
    ],
    "value": [
        legacy_rows_raw, len(legacy_seen_ids), legacy_duplicate_id_values, legacy_invalid_occ_dates,
        legacy_invalid_time_rows, legacy_coordinate_issue_rows,
        legacy_additional_code_counts["Crm Cd 2"], legacy_additional_code_counts["Crm Cd 3"],
        legacy_additional_code_counts["Crm Cd 4"],
    ],
})
display(legacy_quality.style.format({"value": "{:,}"}))

In [ ]:
legacy_years = pd.DataFrame(sorted(legacy_year_counter.items()), columns=["year", "rows"])
legacy_years["source"] = "Legacy records"
legacy_hours = pd.DataFrame(sorted(legacy_hour_counter.items()), columns=["hour", "rows"])
legacy_top_premises = pd.DataFrame(legacy_premise_counter.most_common(15), columns=["premise", "rows"])

print(f"Legacy source rows: {legacy_rows_raw:,}")
print(f"Legacy cleaned rows: {legacy_rows_clean:,}")
print(f"Legacy cells changed by trimming: {int(legacy_trim_change_counts.sum()):,}")
display(legacy_clean_sample)

px.bar(legacy_hours, x="hour", y="rows", title="Legacy records by occurrence hour").show()
px.bar(
    legacy_top_premises.sort_values("rows"), x="rows", y="premise", orientation="h",
    title="Top 15 Legacy premise descriptions",
).show()

In [ ]:
legacy_validation = pd.DataFrame({
    "check": [
        "Row count preserved", "All 28 source columns observed",
        "DR_NO has no missing values", "DR_NO remains unique",
        "Every row reached the cleaning pass", "A cleaned sample was retained",
    ],
    "passed": [
        legacy_rows_clean == legacy_rows_raw,
        len(legacy_source_columns) == 28,
        legacy_missing_id_rows == 0,
        legacy_duplicate_id_values == 0,
        legacy_rows_clean == sum(legacy_year_counter.values()) + legacy_invalid_occ_dates,
        len(legacy_clean_sample) == 5,
    ],
})
display(legacy_validation)
assert legacy_validation["passed"].all(), "Investigate failed Legacy validation checks."

In [ ]:
coverage = pd.concat([legacy_years, nibrs_years_for_comparison], ignore_index=True)
px.bar(
    coverage, x="year", y="rows", color="source", barmode="group",
    title="Coverage by source (row counts are not directly comparable across systems)"
).show()

overlap_evidence = pd.DataFrame({
    "check": [
        "Legacy rows on/after 2024-03-07",
        "NIBRS rows with 2024 occurrence dates",
        "Exact DR_NO / CaseNo text matches",
    ],
    "rows_or_values": [legacy_rows_after_transition, nibrs_2024_rows, identifier_matches],
})
display(overlap_evidence.style.format({"rows_or_values": "{:,}"}))

crosswalk = pd.DataFrame({
    "concept": ["Row identifier", "Case identifier", "Occurrence date", "Occurrence time", "Offense code", "Reporting district", "Premise"],
    "Legacy": ["DR_NO", "Not separately established", "DATE OCC", "TIME OCC", "Crm Cd plus Crm Cd 1–4", "Rpt Dist No", "Premis Cd / Premis Desc"],
    "NIBRS": ["UniqueNIBRNo", "CaseNo", "Date OCC", "Time OCC", "NIBR Code", "RPT Dist No", "Premise Cd / Premise Desc"],
    "safe_to_equate_now": [False, False, True, True, False, "Needs format check", "Needs category check"],
})
display(crosswalk)

### Human decision: why both core files are used but not appended

Both sources belong in Data Fit and cleaning. Using both does not require stacking their rows. The 2024 periods overlap, identifiers do not provide a general join, and offenses are represented differently. For now, the valid output is a validated NIBRS table, validated chunk-level Legacy summaries and sample, plus a comparability report. A combined trend requires a documented offense crosswalk, a common counting unit, a transition boundary, and a rule for possible duplicate events in 2024.

## 8. How the cleaning evidence explains Data Fit

### Why Vision A is only partly supported

Vision A asks an LAPD community-safety manager to prioritize geographic areas and time periods for expanded non-enforcement outreach. Both cleaned core tables contain useful descriptive evidence: occurrence date and time, LAPD area, reporting district, privacy-rounded location, coordinates, offense category, and premise type. Legacy contributes 2020–2024 historical context, while NIBRS contributes the current offense-level structure.

The cleaning process also shows the boundary. Neither the 31 NIBRS columns nor the 28 Legacy columns measures resident priorities, unmet service need, existing outreach locations, program capacity, staffing, cost, participation, or program outcomes. Missingness cleaning cannot create those concepts. A concentration of recorded offenses may also reflect population, activity, reporting, or police-recording differences. Therefore, the files can identify locations for further community review, but cannot independently decide where outreach resources should be allocated or predict whether outreach would reduce offenses.

### Why Vision B fits the core data more directly

Vision B asks a street-lighting manager to prioritize areas with recurring nighttime reported offenses for **on-site assessment**. That decision matches both sources more closely: each has occurrence time, LAPD area, reporting district, privacy-protected location, coordinates, offense categories, and premise descriptions. These fields support separate historical and current geographic screens.

The word “assessment” is crucial. Neither cleaned table contains streetlight locations, illumination, outages, maintenance history, ownership, vegetation, traffic, installation cost, or observed lighting conditions at the time of an offense. Privacy-rounded locations and coordinate-quality flags also prevent pole-level conclusions. The data can create screening lists for inspection; it cannot prescribe a particular lighting improvement.

### Why Vision B fits better than Vision A

Both Visions can use place and time patterns from both files. Vision B fits better because its immediate action is narrower: choose locations for additional physical inspection. The core files observe the place and time inputs needed for that screening. Vision A's immediate action is resource allocation, which depends on several unobserved community and service variables. Legacy improves historical context but does not fill those gaps. Vision B still needs auxiliary streetlight data before any repair or installation decision.

### Why neither Vision supports a causal claim

Cleaning improves consistency; it does not create an experiment. Neither file observes outreach exposure or lighting quality, assigns interventions, or provides a comparison group or before/after outcome design. You may report associations between recorded offense patterns and place or time within each reporting era. You may not claim that limited outreach or inadequate lighting caused an offense, or that either intervention would reduce offenses.

## 9. Refine after reviewing the evidence

Before producing dashboard-ready outputs from either source, write down your answers:

1. Should the primary metric count offense records or distinct cases? Why?
2. What exact hours define nighttime, and is clock time sufficient for the lighting question?
3. Which date range is sufficiently complete within each source, and will the two reporting eras remain separate?
4. Should records with missing or zero coordinates remain in area-level summaries?
5. Which premise categories belong in a lighting-screening view?
6. What auxiliary data is required before recommending outreach or lighting action?
7. Does every chart identify its source, reporting era, and counting unit?

Revise `plan.md` when the team makes a decision, rerun the implementation cells, and verify that the validation results still pass.

In [ ]:
# Both sources were cleaned and validated. Large outputs are intentionally not written by default.
# If the team approves an output schema, add a chunked Parquet writer rather than rebuilding
# a million-row Legacy DataFrame in memory.
if SAVE_CLEAN_DATA:
    print("NIBRS was saved at its checkpoint. Legacy saving requires an approved chunked-output schema.")
else:
    print("No output files written. Both datasets were cleaned and validated in this notebook.")